## 1. Import Libraries

In [ ]:
# Load libraries
import os, sys #os dung de doc duong dan file, tao xoa thu muc kiem tra file ton tai,sys lay tham so dong lenh vaf dung thoat chuong trinh
from IPython import display # dung cho viec hien hinh anh
import numpy as np
import matplotlib.pyplot as plt # ve do thi 
import pandas as pd # dung su ly du lieu dang bang doc , loc du lieu tu csv excel
import seaborn as sns # ve bieu do nang cao
import joblib # luu va load mo hinh machine learning
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder # chuyen du lieu thanh vecto ,text thanh so, du lieu thu tu
from sklearn.preprocessing import MinMaxScaler, StandardScaler # chuan hoa du lieu tu 0 ->1,Chuẩn hóa dữ liệu theo mean = 0 và std = 1
from sklearn.model_selection import train_test_split # chia va train/ test
from sklearn import model_selection
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import warnings
from sklearn.model_selection import cross_val_score
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams['figure.dpi'] = 100

warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
data_path = "eda/data/pima-indians-diabetes.csv"

data_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df_dataset = pd.read_csv(data_path, names=data_names)

In [ ]:
# shape
print(f'+ Shape : {df_dataset.shape}')
# types
print(f'+ Data Types: \n{df_dataset.dtypes}')
# head, tail
print(f'+ Contents: ')
display.display(df_dataset.head(5))
display.display(df_dataset.tail(5))
# info
df_dataset.info()

Ngưỡng sinh lý

In [ ]:
physiological_ranges = {
    "Pregnancies": (0, 15),
    "Glucose": (35, 250),
    "BloodPressure": (40, 180),
    "SkinThickness": (7, 100),
    "Insulin": (15, 900),
    "BMI": (10, 60),
    "DiabetesPedigreeFunction": (0, 2.5),
    "Age": (21, 90)
}

xóa dữ liệu trùng lập

In [ ]:
duplicated = df_dataset.duplicated()

print(f"Số dòng bị trùng: {duplicated.sum()}")

df_clean = df_dataset.drop_duplicates()

print(f"Kích thước sau khi xóa dòng trùng: {df_clean.shape}")

In [ ]:
from xu_ly_du_lieu import xulydulieu

# Áp dụng
df_final = xulydulieu(df_clean,physiological_ranges)

chia dữ liệu 7/3

In [ ]:

X = df_final.drop("Outcome", axis=1)
y = df_final["Outcome"]
seed=1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# 1. Dữ liệu thô (Raw) - Bạn đã có sẵn
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

# 2. Xử lý chuẩn hóa Min/Max
from sklearn.preprocessing import MinMaxScaler
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# 3. Xử lý chuẩn hóa Standard (Z-score)
from sklearn.preprocessing import StandardScaler
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)

In [ ]:

X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=seed, stratify=y_train
)

print("Train sub:", X_train_sub.shape)
print("Validation:", X_valid.shape)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print("--- 1. ĐÁNH GIÁ BASELINE (THAM SỐ MẶC ĐỊNH) ---")
# Khởi tạo mô hình KNN MẶC ĐỊNH (k=5)
knn_baseline = KNeighborsClassifier()

# Chạy trên dữ liệu Thô (Raw)
knn_baseline.fit(X_train_raw, y_train)
acc_raw_knn = accuracy_score(y_test, knn_baseline.predict(X_test_raw))

# Chạy trên dữ liệu Min/Max
knn_baseline.fit(X_train_minmax, y_train)
acc_minmax_knn = accuracy_score(y_test, knn_baseline.predict(X_test_minmax))

# Chạy trên dữ liệu Standard
knn_baseline.fit(X_train_std, y_train)
acc_std_knn = accuracy_score(y_test, knn_baseline.predict(X_test_std))

# Tạo bảng tóm tắt
results_knn = pd.DataFrame({
    'Loại dữ liệu': ['Dữ liệu Thô (Raw)', 'Chuẩn hóa Min/Max', 'Chuẩn hóa Standard'],
    'Accuracy (Tập Test)': [acc_raw_knn, acc_minmax_knn, acc_std_knn]
})

display.display(results_knn)

# Vẽ biểu đồ trình diễn
plt.figure(figsize=(8, 5))
sns.barplot(x='Loại dữ liệu', y='Accuracy (Tập Test)', data=results_knn, palette='viridis')
plt.title('SO SÁNH HIỆU SUẤT KNN (MẶC ĐỊNH) TRÊN CÁC LOẠI DỮ LIỆU', fontsize=14, fontweight='bold')
plt.ylabel('Độ chính xác (Accuracy)')
plt.ylim(0.60, 0.85)
for index, row in results_knn.iterrows():
    plt.text(index, row['Accuracy (Tập Test)'] + 0.005, f"{row['Accuracy (Tập Test)']:.4f}", color='black', ha="center")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix

print("--- 2. TINH CHỈNH THAM SỐ CHO KNN (DÙNG DỮ LIỆU STANDARD) ---")

# 1. Liệt kê các tham số muốn thử
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15], 
    'weights': ['uniform', 'distance'],      
    'metric': ['euclidean', 'manhattan']     
}

# 2. Khởi tạo mô hình và bộ dò tìm 
knn_base = KNeighborsClassifier()
grid_search = GridSearchCV(estimator=knn_base, param_grid=param_grid, cv=10, scoring='accuracy')

# 3. Huấn luyện để dò tìm
grid_search.fit(X_train_std, y_train)

# 4. In ra bộ tham số tốt nhất
best_knn = grid_search.best_estimator_
print("Tham số KNN tốt nhất tìm được:", grid_search.best_params_)

# 5. Đánh giá mô hình đã tinh chỉnh trên tập Test
y_pred_tuned = best_knn.predict(X_test_std)
acc_tuned = accuracy_score(y_test, y_pred_tuned)

print("Test Accuracy (Sau tinh chỉnh):", acc_tuned)
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
print("Ma trận nhầm lẫn:\n", cm_tuned)

K-flod

In [ ]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import cross_val_score

print("--- 3. XUẤT KẾT QUẢ VÀ LƯU MÔ HÌNH ---")

# Áp dụng K-fold với cv=10 cho MÔ HÌNH TỐT NHẤT
scores_tuned = cross_val_score(best_knn, X_train_std, y_train, cv=10)

# 1. Tạo bảng tóm tắt so sánh Baseline vs Tinh chỉnh (Giống hệt form LDA)
summary_df = pd.DataFrame({
    "Phiên bản Mô hình": ["Baseline (Tham số mặc định)", "Tuned (Đã tinh chỉnh)"],
    "Test Accuracy": [acc_std_knn, acc_tuned], # Đọ điểm Standard của Ô 1 với điểm Tuned của Ô 2
    "Bộ tham số": ["Mặc định", str(grid_search.best_params_)]
})

# 2. Ma trận nhầm lẫn & KFold của mô hình Tinh chỉnh
cm_df = pd.DataFrame(cm_tuned)
kfold_df = pd.DataFrame(scores_tuned, columns=["KFold Scores"])

# ===== XUẤT FILE EXCEL =====
folder_path = "ket_qua"  
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

file_path = os.path.join(folder_path, "ket_qua_modelKNeighbors.xlsx")

with pd.ExcelWriter(file_path) as writer:
    # Sheet 1: So sánh 3 loại dữ liệu
    results_knn.to_excel(writer, sheet_name="Baseline_3_Loai_Data", index=False) 
    # Sheet 2: Bảng đọ sức Baseline và Tuned
    summary_df.to_excel(writer, sheet_name="So_Sanh_Tuning", index=False)       
    # Sheet 3 & 4: Chi tiết
    cm_df.to_excel(writer, sheet_name="Confusion_Matrix_Tuned", index=False)
    kfold_df.to_excel(writer, sheet_name="KFold_Tuned", index=False)

print(f"Đã xuất báo cáo xuất sắc vào: {file_path}")

# ===== LƯU 2 MÔ HÌNH (Đúng chuẩn Mục 8 Đề bài) =====
joblib.dump(knn_baseline, 'knn_baseline_pima.pkl') # Lưu mô hình gốc
joblib.dump(best_knn, 'knn_tuned_pima.pkl')        # Lưu mô hình đã tinh chỉnh
print("Đã lưu cả 2 file mô hình (.pkl) thành công!")

In [ ]:
!jupyter nbconvert --to html K-neighbors.ipynb